In [31]:
import os
import sys
import requests
import datetime as dt
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from pathlib import Path

import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.rci import Rci
from Modules.show_plot import ShowPlot

In [32]:
# APIの接続先（notebookコンテナでは通常 http://api:8000 が設定される）
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
# データを取得する関数
sys.path.append("/workspace/notebook")

In [33]:
# S&P 500の時系列データを登録
code = "^GSPC"
market = ""
start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")

"""
API_BASE_URLpost_payload = {
    "code": code,
    "market": market,
    "start": start,
    "end": end
}

# /api/v1/stock_price/
post_url = f"{API_BASE_URL}/api/v1/stock_price/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""


'\nAPI_BASE_URLpost_payload = {\n    "code": code,\n    "market": market,\n    "start": start,\n    "end": end\n}\n\n# /api/v1/stock_price/\npost_url = f"{API_BASE_URL}/api/v1/stock_price/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

### S&P500の時系列データを取得

In [34]:
# S&P 500の時系列データを登録
sp500 = "^GSPC"
market = None
start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")
"""
post_payload = {
    "code": sp500,
    "market": market,
    "start": start,
    "end": end
}

# /api/v1/stock_price/
post_url = f"{API_BASE_URL}/api/v1/stock_price/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""

'\npost_payload = {\n    "code": sp500,\n    "market": market,\n    "start": start,\n    "end": end\n}\n\n# /api/v1/stock_price/\npost_url = f"{API_BASE_URL}/api/v1/stock_price/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

In [35]:
get_url = f"{API_BASE_URL}/api/v1/time_series_data/stock/"
get_params = {
    "code": sp500,
    "market": market,
    "start": start,
    "end": end
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_sp = pd.DataFrame(results)
    print("取得件数:", len(df_sp))
    display(df_sp)
else:
    print("GET response:", get_response.json())


GET URL: http://api:8000/api/v1/time_series_data/stock/
GET status: 200
取得件数: 6895


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1125268,^GSPC,,1998-11-17,1139.319946,1151.709961,1129.670044,1135.869995,705200000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1125269,^GSPC,,1998-11-18,1144.479980,1144.520020,1133.069946,1139.319946,652510000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1125270,^GSPC,,1998-11-19,1152.609985,1155.099976,1144.420044,1144.479980,671000000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1125271,^GSPC,,1998-11-20,1163.550049,1163.550049,1152.609985,1152.609985,721200000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1125272,^GSPC,,1998-11-23,1188.209961,1188.209961,1163.550049,1163.550049,774100000,1147.165991,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6890,1132158,^GSPC,,2026-04-13,6886.240234,6887.000000,6790.020020,6806.470215,4785840000,6757.138086,...,6760.913781,6515.038290,True,NaN,NaN,NaN,NaN,NaN,NaN,False
6891,1132159,^GSPC,,2026-04-14,6967.379883,6969.419922,6905.169922,6910.200195,5032380000,6818.792090,...,6780.011707,6512.772395,True,NaN,NaN,NaN,NaN,NaN,NaN,False
6892,1132160,^GSPC,,2026-04-15,7022.950195,7026.240234,6967.129883,6978.169922,5278610000,6863.554102,...,6799.610157,6507.702733,True,NaN,NaN,NaN,NaN,NaN,NaN,False
6893,1132161,^GSPC,,2026-04-16,7041.279785,7051.229980,7008.520020,7037.779785,5173650000,6914.372070,...,6825.927192,6501.200893,True,NaN,NaN,NaN,NaN,NaN,NaN,False


### 米国長期金利(10年物)

In [36]:
tnx = "^TNX"
market = None
start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")
"""
post_payload = {
    "code": tnx,
    "market": market,
    "start": start,
    "end": end
}

# /api/v1/stock_price/
post_url = f"{API_BASE_URL}/api/v1/stock_price/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""

'\npost_payload = {\n    "code": tnx,\n    "market": market,\n    "start": start,\n    "end": end\n}\n\n# /api/v1/stock_price/\npost_url = f"{API_BASE_URL}/api/v1/stock_price/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

In [37]:
get_url = f"{API_BASE_URL}/api/v1/time_series_data/stock/"
get_params = {
    "code": tnx,
    "market": market,
    "start": start,
    "end": end
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_tnx = pd.DataFrame(results)

    print("取得件数:", len(df_tnx))
    display(df_tnx)
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/time_series_data/stock/
GET status: 200
取得件数: 6887


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1139058,^TNX,None,1998-11-17,4.850,4.862,4.770,4.826,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1139059,^TNX,None,1998-11-18,4.842,4.898,4.806,4.858,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1139060,^TNX,None,1998-11-19,4.846,4.854,4.818,4.850,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1139061,^TNX,None,1998-11-20,4.810,4.830,4.790,4.830,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1139062,^TNX,None,1998-11-23,4.830,4.834,4.778,4.802,0,4.8332,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6882,1145940,^TNX,None,2026-04-13,4.297,4.351,4.297,4.349,0,4.3048,...,4.372770,4.216830,True,NaN,NaN,NaN,NaN,NaN,NaN,False
6883,1145941,^TNX,None,2026-04-14,4.256,4.305,4.252,4.297,0,4.2952,...,4.373662,4.225858,False,NaN,4.29976,NaN,NaN,NaN,NaN,False
6884,1145942,^TNX,None,2026-04-15,4.282,4.293,4.264,4.272,0,4.3000,...,4.371202,4.239518,False,NaN,NaN,NaN,NaN,NaN,NaN,False
6885,1145943,^TNX,None,2026-04-16,4.309,4.317,4.266,4.276,0,4.2958,...,4.370106,4.248534,False,NaN,NaN,NaN,NaN,NaN,NaN,False


### 金の時系列データを登録

In [38]:
gold = "GC=F"
start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")
"""
post_payload = {
    "code": gold,
    "market": None,
    "start": "1999-01-01",
    "end": "2026-04-18"
}

# /api/v1/commodity_data/
post_url = f"{API_BASE_URL}/api/v1/commodity_data/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""

'\npost_payload = {\n    "code": gold,\n    "market": None,\n    "start": "1999-01-01",\n    "end": "2026-04-18"\n}\n\n# /api/v1/commodity_data/\npost_url = f"{API_BASE_URL}/api/v1/commodity_data/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

In [39]:
# /api/v1/time_series_data/commodity/
get_url = f"{API_BASE_URL}/api/v1/time_series_data/commodity/"
get_params = {
    "code": gold,
    "market": None,
    "start": start,
    "end": end
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_gold = pd.DataFrame(results)
    print("取得件数:", len(df_gold))
    display(df_gold)
else:
    print("GET response:", get_response.json())
    

GET URL: http://api:8000/api/v1/time_series_data/commodity/
GET status: 200
取得件数: 6431


,id,commodity_id,commodity_code,commodity_market,date,open,high,low,close,adj_close,...,rci9,rci26,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,9,14,GC=F,None,2000-08-30,273.899994,273.899994,273.899994,273.899994,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,10,14,GC=F,None,2000-08-31,278.299988,278.299988,274.799988,274.799988,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,11,14,GC=F,None,2000-09-01,277.000000,277.000000,277.000000,277.000000,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,12,14,GC=F,None,2000-09-05,275.799988,275.799988,275.799988,275.799988,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,13,14,GC=F,None,2000-09-06,274.200012,274.200012,274.200012,274.200012,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6426,6435,14,GC=F,None,2026-04-13,4742.399902,4742.399902,4704.000000,4704.000000,None,...,40.000000,-59.726496,False,NaN,NaN,NaN,NaN,NaN,NaN,False
6427,6436,14,GC=F,None,2026-04-14,4825.000000,4841.600098,4770.100098,4770.100098,None,...,40.000000,-51.111111,False,NaN,NaN,NaN,NaN,NaN,NaN,False
6428,6437,14,GC=F,None,2026-04-15,4800.000000,4843.600098,4798.000000,4843.600098,None,...,48.333333,-39.555556,True,4754.92002,NaN,NaN,NaN,NaN,NaN,False
6429,6438,14,GC=F,None,2026-04-16,4785.399902,4810.899902,4785.399902,4810.899902,None,...,81.666667,-28.000000,True,NaN,NaN,NaN,NaN,NaN,NaN,False


### VIX指数

In [40]:
vix = "^VIX"
start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")
"""
post_payload = {
    "code": vix,
    "market": None,
    "start": "1999-01-01",
    "end": "2026-04-18"
}

# /api/v1/commodity_data/
post_url = f"{API_BASE_URL}/api/v1/commodity_data/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())
"""

'\npost_payload = {\n    "code": vix,\n    "market": None,\n    "start": "1999-01-01",\n    "end": "2026-04-18"\n}\n\n# /api/v1/commodity_data/\npost_url = f"{API_BASE_URL}/api/v1/commodity_data/"\npost_response = requests.post(post_url, json=post_payload, timeout=60)\n\nprint("POST URL:", post_url)\nprint("POST status:", post_response.status_code)\nprint("POST response:", post_response.json())\n'

In [45]:
# /api/v1/time_series_data/stock/
get_url = f"{API_BASE_URL}/api/v1/time_series_data/stock/"
get_params = {
    "code": vix,
    "market": None,
    "start": start,
    "end": end
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_vix = pd.DataFrame(results)
    print("取得件数:", len(df_vix))
    display(df_vix)
else:
    print("GET response:", get_response.json())
    

GET URL: http://api:8000/api/v1/time_series_data/stock/
GET status: 200
取得件数: 627


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1134479,^VIX,None,2008-02-05,28.240000,28.490000,27.200001,27.200001,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1134480,^VIX,None,2008-02-06,28.969999,29.309999,27.040001,27.750000,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1134481,^VIX,None,2008-02-07,27.660000,29.700001,26.780001,29.510000,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1134482,^VIX,None,2008-02-08,28.010000,28.889999,27.219999,27.980000,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1134483,^VIX,None,2008-02-11,27.600000,29.570000,27.320000,29.139999,0,28.316000,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
622,1135101,^VIX,None,2010-07-26,22.730000,24.610001,22.700001,24.370001,0,24.898001,...,30.268157,23.960643,False,NaN,NaN,NaN,NaN,NaN,NaN,False
623,1135102,^VIX,None,2010-07-27,23.190001,23.570000,21.860001,21.889999,0,23.806001,...,30.287391,23.860610,False,NaN,NaN,NaN,NaN,NaN,-49.094017,False
624,1135103,^VIX,None,2010-07-28,24.250000,24.540001,22.240000,23.930000,0,23.870000,...,30.276648,23.774552,False,NaN,NaN,NaN,NaN,-67.5,NaN,True
625,1135104,^VIX,None,2010-07-29,24.129999,25.540001,23.040001,23.400000,0,23.676000,...,30.212284,23.556517,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [46]:
start = dt.datetime(2025, 1, 1)
end = dt.datetime.now().strftime("%Y-%m-%d")
df_sp_tmp = df_sp.copy()

# S&P500
if "date" not in df_sp_tmp.columns:
    df_sp_tmp = df_sp_tmp.reset_index()

df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
df_sp_tmp = df_sp_tmp.set_index("date")
df_sp_tmp = df_sp_tmp.loc[start:end]

# 金価格
df = df_gold.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df = df.loc[start:end]

# インデックスを揃えて結合
df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)

show_plot = ShowPlot()
fig = show_plot.create_basic_chart(
    df=df.reset_index(),  # create_basic_chartはdate列を使うため
    code="GC=F",
    name="Gold",
    start=start.strftime("%Y-%m-%d"),
    end=end
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA5_SP"],
        name="SP_MA5",
        line={"color": "orange", "width": 1.2},
        yaxis="y1"
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA25_SP"],
        name="SP_MA25",
        line={"color": "green", "width": 1.2},
        yaxis="y1"
    )
)
fig.show()

In [47]:
start = dt.datetime(2008, 12, 1)
end = dt.datetime(2010, 7, 31)
df_sp_tmp = df_sp.copy()

# S&P500
if "date" not in df_sp_tmp.columns:
    df_sp_tmp = df_sp_tmp.reset_index()

df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
df_sp_tmp = df_sp_tmp.set_index("date")
df_sp_tmp = df_sp_tmp.loc[start:end]

# 金価格
df = df_gold.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df = df.loc[start:end]

# インデックスを揃えて結合
df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)

show_plot = ShowPlot()
fig = show_plot.create_basic_chart(
    df=df.reset_index(),  # create_basic_chartはdate列を使うため
    code="GC=F",
    name="Gold",
    start=start.strftime("%Y-%m-%d"),
    end=end
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA5_SP"],
        name="SP_MA5",
        line={"color": "orange", "width": 1.2},
        yaxis="y1"
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA25_SP"],
        name="SP_MA25",
        line={"color": "green", "width": 1.2},
        yaxis="y1"
    )
)
fig.show()

In [48]:
# Gold
df_gold_close = df_gold.set_index("date")[["close"]].rename(columns={"close": "Gold"})
# SP500
df_sp_close = df_sp.set_index("date")[["close"]].rename(columns={"close": "SP500"})
# TNX
df_tnx_close = df_tnx.set_index("date")[["close"]].rename(columns={"close": "TNX"})
# VIX
df_vix_close = df_vix.set_index("date")[["close"]].rename(columns={"close": "VIX"})
# 結合
df_research = pd.concat([df_gold_close, df_sp_close, df_tnx_close, df_vix_close], axis=1).dropna()
df_research

,Gold,SP500,TNX,VIX
date,,,,
2008-02-05,905.099976,1380.280029,3.624,27.200001
2008-02-06,887.900024,1339.479980,3.583,27.750000
2008-02-07,900.599976,1324.010010,3.573,29.510000
2008-02-08,915.299988,1336.880005,3.707,27.980000
2008-02-11,920.700012,1331.920044,3.665,29.139999
...,...,...,...,...
2010-07-26,1183.000000,1102.890015,2.989,24.370001
2010-07-27,1158.000000,1117.359985,3.041,21.889999
2010-07-28,1160.400024,1112.839966,3.036,23.930000


In [49]:
df_ret = np.log(df_research).diff().dropna()
df_ret

,Gold,SP500,TNX,VIX
date,,,,
2008-02-06,-0.019186,-0.030005,-0.011378,0.020019
2008-02-07,0.014202,-0.011616,-0.002795,0.061493
2008-02-08,0.016191,0.009674,0.036817,-0.053239
2008-02-11,0.005882,-0.003717,-0.011395,0.040622
2008-02-12,0.000326,0.006458,0.005171,-0.092705
...,...,...,...,...
2010-07-26,-0.003965,0.009767,0.005031,-0.017087
2010-07-27,-0.021359,0.013035,0.017247,-0.107323
2010-07-28,0.002070,-0.004053,-0.001646,0.089103


In [50]:
from statsmodels.tsa.stattools import adfuller

for col in df_ret.columns:
    print(col, adfuller(df_ret[col])[1])  # p-value

Gold 3.230387097664157e-09
SP500 1.2334213452245363e-27
TNX 0.00012848791283589375
VIX 9.46274401706293e-12


In [51]:
from statsmodels.tsa.api import VAR

model = VAR(df_ret)
results = model.fit(maxlags=5, ic='aic')
print(results.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sun, 19, Apr, 2026
Time:                     08:00:03
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                   -29.3757
Nobs:                     624.000    HQIC:                  -29.5322
Log likelihood:           5739.41    FPE:                1.35250e-13
AIC:                     -29.6317    Det(Omega_mle):     1.27720e-13
--------------------------------------------------------------------
Results for equation Gold
              coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------
const            0.000367         0.000638            0.575           0.565
L1.Gold          0.049962         0.039926            1.251           0.211
L1.SP500         0.014394         0.046195            0.312           0.755
L1.TN

/workspace/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [52]:
"""
# 銘柄リスト
miners = [
    # Undervalued Producers
    { "name": "Avino Silver & Gold Mines", "code": "ASM", "market": "TO" },
    { "name": "Americas Gold & Silver", "code": "USAS", "market": "" },
    { "name": "Santacruz Silver Mining", "code": "SCZ", "market": "V" },
    { "name": "Guanajuato Silver", "code": "GSVR", "market": "V" },
    { "name": "Aya Gold & Silver", "code": "AYA", "market": "TO" },
    { "name": "Andean Precious Metals", "code": "APM", "market": "TO" },

    # Near-Term Producers
    { "name": "Silverco Mining", "code": "SICO", "market": "V" },
    { "name": "Silver Mountain Resources", "code": "AGMR", "market": "V" },
    { "name": "Silver Storm Mining", "code": "SVRS", "market": "V" },
    { "name": "Andean Silver", "code": "ASL", "market": "AX" },

    # Developers
    { "name": "Discovery Silver", "code": "DSV", "market": "TO" },
    { "name": "GoGold Resources", "code": "GGD", "market": "TO" },
    { "name": "Vizsla Silver", "code": "VZLA", "market": "" },
    { "name": "Vizsla Silver (TSX)", "code": "VZLA", "market": "TO" },
    { "name": "Aftermath Silver", "code": "AAGFF", "market": "" },
    { "name": "Blackrock Silver", "code": "BKRRF", "market": "" },
    { "name": "Silver Tiger Metals", "code": "SLVR", "market": "V" },
    { "name": "Highlander Silver", "code": "HSLV", "market": "TO" },
    { "name": "Outcrop Silver & Gold", "code": "OCGSF", "market": "" },
    { "name": "Silver X Mining", "code": "AGX", "market": "V" },

    # Optionality
    { "name": "Chesapeake Gold", "code": "CHPGF", "market": "" },
    { "name": "Eloro Resources", "code": "ELO", "market": "TO" },
    { "name": "Hycroft Mining", "code": "HYMC", "market": "" },
    { "name": "Southern Silver Exploration", "code": "SSVFF", "market": "" }
]

for miner in miners:
    # /api/v1/search/
    post_url = f"{API_BASE_URL}/api/v1/search/"
    post_data = {
        "code": miner["code"],
        "market": miner["market"],
        "name": miner["name"]
    }

    post_response = requests.post(post_url, json=post_data, timeout=60)
    if post_response.status_code == 200:
        results = post_response.json()
    else:
        results = {"name": None, "code": None}
    print(f"POST status: {post_response.status_code} {miner['name']} ", results["code"])
"""


'\n# 銘柄リスト\nminers = [\n    # Undervalued Producers\n    { "name": "Avino Silver & Gold Mines", "code": "ASM", "market": "TO" },\n    { "name": "Americas Gold & Silver", "code": "USAS", "market": "" },\n    { "name": "Santacruz Silver Mining", "code": "SCZ", "market": "V" },\n    { "name": "Guanajuato Silver", "code": "GSVR", "market": "V" },\n    { "name": "Aya Gold & Silver", "code": "AYA", "market": "TO" },\n    { "name": "Andean Precious Metals", "code": "APM", "market": "TO" },\n\n    # Near-Term Producers\n    { "name": "Silverco Mining", "code": "SICO", "market": "V" },\n    { "name": "Silver Mountain Resources", "code": "AGMR", "market": "V" },\n    { "name": "Silver Storm Mining", "code": "SVRS", "market": "V" },\n    { "name": "Andean Silver", "code": "ASL", "market": "AX" },\n\n    # Developers\n    { "name": "Discovery Silver", "code": "DSV", "market": "TO" },\n    { "name": "GoGold Resources", "code": "GGD", "market": "TO" },\n    { "name": "Vizsla Silver", "code": "VZLA",

In [53]:
"""
search_url = f"{API_BASE_URL}/api/v1/search/"
registration_api_url = f"{API_BASE_URL}/api/v1/stock_price/"

start = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")

# DBに銘柄を登録
for miner in miners:
    # /api/v1/search/
    # 銘柄を検索
    search_data = {
        "code": miner["code"],
        "market": miner["market"],
        "name": miner["name"]
    }
    post_response = requests.post(search_url, json=search_data, timeout=60)
    if post_response.status_code == 200:
        results = post_response.json()
    else:
        results = {"name": None, "code": None}
    # 銘柄を登録
    post_payload = {
        "code": results["code"],
        "market": search_data["market"],
        "start": start,
        "end": end
    }

    post_response = requests.post(registration_api_url, json=post_payload, timeout=60)
    print(f"POST status: {post_response.status_code} {miner['name']} ", results["code"])
    """

'\nsearch_url = f"{API_BASE_URL}/api/v1/search/"\nregistration_api_url = f"{API_BASE_URL}/api/v1/stock_price/"\n\nstart = dt.datetime(1999, 1, 1).strftime("%Y-%m-%d")\nend = dt.datetime.now().strftime("%Y-%m-%d")\n\n# DBに銘柄を登録\nfor miner in miners:\n    # /api/v1/search/\n    # 銘柄を検索\n    search_data = {\n        "code": miner["code"],\n        "market": miner["market"],\n        "name": miner["name"]\n    }\n    post_response = requests.post(search_url, json=search_data, timeout=60)\n    if post_response.status_code == 200:\n        results = post_response.json()\n    else:\n        results = {"name": None, "code": None}\n    # 銘柄を登録\n    post_payload = {\n        "code": results["code"],\n        "market": search_data["market"],\n        "start": start,\n        "end": end\n    }\n\n    post_response = requests.post(registration_api_url, json=post_payload, timeout=60)\n    print(f"POST status: {post_response.status_code} {miner[\'name\']} ", results["code"])\n    '

In [ ]:
code = "HYMC"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# /api/v1/time_series_data/stock/
get_url = f"{API_BASE_URL}/api/v1/time_series_data/stock/"
get_params = {
    "code": code,
    "market": None,
    "start": start,
    "end": end
}
# APIからデータを取得
get_response = requests.get(get_url, params=get_params, timeout=60)
get_json = get_response.json()
results = get_json.get("results", [])
df_hymc = pd.DataFrame(results)

# S&P500を統合
df_sp_tmp = df_sp.copy()
if "date" not in df_sp_tmp.columns:
    df_sp_tmp = df_sp_tmp.reset_index()

df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
df_sp_tmp = df_sp_tmp.set_index("date")
df_sp_tmp = df_sp_tmp.loc[start:end]

# 金価格
df_gold_tmp = df_gold.copy()
if "date" not in df_gold_tmp.columns:
    df_gold_tmp = df_gold_tmp.reset_index()
df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
df_gold_tmp = df_gold_tmp.set_index("date")
df_gold_tmp = df_gold_tmp.loc[start:end]

# 金価格
df = df_hymc.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df = df.loc[start:end]


# インデックスを揃えて結合
df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)

show_plot = ShowPlot()
fig = show_plot.create_basic_chart(
    df=df.reset_index(),
    code="HYMC",
    name="HYMC",
    start=start,
    end=end
)
# ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
fig.update_layout(
    yaxis=dict(
        title="HYMC Price",
        side="left"
    ),
    yaxis2=dict(
        title="SP500 / GOLD",
        overlaying="y",
        side="right"
    )
)

# --- SP500（右軸） ---
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA5_SP"],
        name="SP_MA5",
        line={"color": "blue", "width": 1.2},
        yaxis="y2"   # ★ 右軸
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA25_SP"],
        name="SP_MA25",
        line={"color": "gray", "width": 1.2},
        yaxis="y2"   # ★ 右軸
    )
)

# --- GOLD（右軸） ---
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA5_GOLD"],
        name="GOLD_MA5",
        line={"color": "orange", "width": 1.2},
        yaxis="y2"   # ★ 右軸
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["MA25_GOLD"],
        name="GOLD_MA25",
        line={"color": "green", "width": 1.2},
        yaxis="y2"   # ★ 右軸
    )
)

fig.show()